# Chapter — Ising, duality and contextual kernels

**Typed textbook form:** `SIGIL[chapter] · PLURALTYPE[representations] · QUNO[alternatives] · CODEBOOK[witness]`

This chapter is an executable bridge for Physics, Mathematics and Computer Science. It follows: mathematical kernel → observables → executable model → experiment → interpretation → reproducibility.


## 0. Runtime and optional SIGIL4Py

The scientific path only needs NumPy/Matplotlib. `sigil4py` is optional and must not be required to reproduce the physics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import sigil4py  # optional public semantic/replay layer
    SIGIL4PY_AVAILABLE = True
except ImportError:
    SIGIL4PY_AVAILABLE = False

from ising_duality_core import *
from toy_polykategory import *
print('sigil4py available:', SIGIL4PY_AVAILABLE)


## 1. Classical Ising / Onsager / Metropolis

Mathematical kernel: $H=-J\sum_{\langle i,j\rangle}s_i s_j$ with $s_i\in\{-1,+1\}$. Metropolis is a **local sampling dynamics**. It is not itself a duality transformation.


In [ ]:
Tc = onsager_critical_temperature()
print('Onsager Tc/J =', Tc)
rows=[]
for T in (1.6, 2.0, Tc, 2.8, 3.4):
    m=Ising2D(n=20,J=1.0,beta=1.0/T,seed=11)
    s=m.sample(burn_in=80,sweeps=220,thin=4)
    rows.append((T,s['energy_per_spin'].mean(),np.abs(s['magnetization']).mean()))
rows


## 2. Kramers–Wannier duality

The dual coupling satisfies $\sinh(2K)\sinh(2K^\star)=1$. At the square-lattice critical point, $K_c=K_c^\star$.


In [ ]:
Kc=1/Tc
print(Kc, kramers_wannier_dual_coupling(Kc), np.isclose(Kc,kramers_wannier_dual_coupling(Kc)))


## 3. Hopfield as an Ising-like learning model

Hopfield uses an energy $E=-\frac12s^TWs$. The common energy language is useful, but the Hopfield couplings/learning semantics are not the Onsager square lattice.


In [ ]:
patterns=np.array([[1,1,1,-1,-1,-1],[1,-1,1,-1,1,-1]])
W=hebbian_weights(patterns)
noisy=patterns[0].copy(); noisy[[1,4]]*=-1
recalled=hopfield_async_update(noisy,W,seed=3,sweeps=4)
print('energies:',hopfield_energy(noisy,W),hopfield_energy(recalled,W),'recalled:',recalled)


## 4. Quantum Ising chart

Use the transverse-field Ising Hamiltonian $H=-J\sum_i Z_iZ_{i+1}-h\sum_iX_i$ as a small exact-diagonalization bridge to QML.


In [ ]:
for h in (0.5,1.0,1.5):
    print(h, tfim_ground_state_observables(n=6,J=1.0,h=h))


## 5. Genus-2 sectors

$H_1(\Sigma_2,\mathbb Z_2)\cong\mathbb Z_2^4$, with basis `(a1,b1,a2,b2)`. Toggling a homology bit is sector bookkeeping; energy neutrality depends on the actual Hamiltonian.


In [ ]:
sectors=genus2_zero_two_sectors()
print('number of Z2 sectors:',len(sectors))
print((0,0,0,0),'--a1-->',logical_loop_update((0,0,0,0),'a1'))


## 6. Montonen–Olive / S-duality as an advanced comparison

The pedagogical map `tau -> -1/tau` represents a strong/weak electric–magnetic exchange. This codebook does **not** simulate $N=4$ supersymmetric Yang–Mills, and S-duality is not a Monte Carlo update rule.


In [ ]:
tau=0.25+2.0j
print('tau =',tau,'S(tau)=',montonen_olive_s_transform(tau))


## 7. Contextual kernel

We convert observables and dual coordinates into a feature vector and construct a PSD Gram matrix. The kernel is a learning representation; it does not by itself establish quantum advantage.


In [ ]:
X=[]
for T,e,m in rows:
    X.append(contextual_features(temperature=T,energy_per_spin=e,magnetization=m,dual_coupling=kramers_wannier_dual_coupling(1/T)))
X=np.asarray(X)
K=contextual_kernel(X,gamma=1.0)
print('kernel eigenvalues:',assert_psd(K))
plt.imshow(K); plt.colorbar(); plt.title('Contextual Ising kernel');


## 8. Localization as a plural typed chart

`QUNO[HIGGS, ANDERSON, AUBRY_ANDRE, MOTT, MODE_LOCK, OTHER]` preserves distinct mechanisms. In particular, Anderson localization comes from disorder/interference, whereas Aubry–André uses quasiperiodic modulation and is self-dual at its critical ratio in the standard 1D model.


In [ ]:
print(localization_bundle())
for lam in (1.0,2.0,3.0):
    H=aubry_andre_hamiltonian(64,J=1.0,lam=lam)
    vals,vecs=np.linalg.eigh(H)
    print('lambda/J=',lam,'IPR(mid)=',inverse_participation_ratio(vecs[:,len(vals)//2]))


## 9. Oscillator chart: circle map / Arnold tongues / Kuramoto

Circle-map mode locking is a different dynamical mechanism from lattice localization. Kuramoto synchronization is another distinct oscillator model. They are connected here by a typed comparison interface, not collapsed into one model.


In [ ]:
for Kcmap in (0.2,0.8,1.2):
    print('K=',Kcmap,'rotation=',circle_rotation_number(0.1,0.5,Kcmap,steps=3000,burn_in=300))
theta=np.linspace(0,2*np.pi,12,endpoint=False)
omega=np.linspace(-0.2,0.2,12)
print('Kuramoto |r|=',abs(kuramoto_order_parameter(theta)),'rhs sample=',kuramoto_rhs(theta,omega,1.0)[:3])


## 10. SIGILITAS / QUAZRIS / QUNO toy polykategory

This is an **allegorical/semantic teaching layer**. `exchange_J` explicitly swaps syntax/semantics in the toy container and records a localization tag. `QUAZRIS` stores a witnessed dual projection; it does not assert physical equivalence without an external theorem/witness.


In [ ]:
x=SIGILITAS(syntax='Ising spins',semantics={'observable':'magnetization'},provenance=('codebook',))
xJ,w=exchange_J(x,localization=Localization.OTHER,note='pedagogical syntax/semantics exchange')
qx=quazris_twist(x,dual_syntax='domain-wall variables',dual_semantics={'chart':'dual'},witness='typed teaching projection')
print(xJ); print(w); print(qx)
print(self_dual_closed_subtheory_example())


## 11. Exercise / assessment

1. Reproduce the Ising observables across temperature and discuss finite-size effects.
2. Numerically verify the Kramers–Wannier self-dual point.
3. Compare Hopfield recall with the local Ising simulation and explain what is structurally shared and what is not.
4. Compare a small TFIM scan with the classical transition language.
5. Add a genus-2 sector feature and inspect how it changes the contextual kernel.
6. Choose one plural chart (Anderson/Aubry–André, circle map/Kuramoto, or lattice-light) and justify the typed map without claiming an unsupported physical duality.
7. State explicitly what evidence would be needed before claiming quantum advantage.


## Reproducibility witness

Record Python version, package versions, random seeds, lattice sizes, temperatures/couplings, codebook commit, and all non-default parameters. The public codebook is valid only when the execution path is replayable.
